In [ ]:
import utils

In [ ]:
#First let's generate the a less-than-interesting coloring of K_3 just to get a taste of what we're doing. Start simple and grow.
red = [(1,2),(0,2)]
blue = [e for e in combinations(range(3),2) if e not in red]

r_coloring_K_n(3,edge_coloring=[red,blue], node_size=2000, labels={0:"Calvin", 1:"Hobbes", 2:"Scooby"}, palette=["red","blue"], title=" ")

In [ ]:
#Now let's generate all colorings and select one from it that meets our criteria for K_4
def generate_all_colorings(N):
    edges = list(combinations(range(N),2))
    partitions = []
    total = len(edges)
    for i in range(1,2**(total-1)):
        red = {edges[j] for j in range(total) if i & (1 << j)} #doing a binary selection if bit j is set in i
        blue = {edges[j] for j in range(total) if not(i & (1<<j))} | {edges[-1]} #grabs everything left over
        partitions.append([red,blue])
    return partitions
    
n_coloring = generate_all_colorings(4)

for i in range(len(n_coloring)):
    if not monochromatic_triangle_check(n_coloring[i]):
        print(n_coloring[i])
        print(i)
        break

In [ ]:
#Now we have the index for K_4, we can take a look at what that coloring looks like
all_colorings = generate_all_colorings(4)

red = list(all_colorings[11][0])
blue = list(all_colorings[11][1])

r_coloring_K_n(4,edge_coloring=[red,blue], node_size=2000, labels={0:"Hobbes", 1:"Calvin", 2:"Scooby",3:"Jerry"}, palette=["red","blue"])

In [ ]:
#Same thing for K_5
n_coloring = generate_all_colorings(5)
for i in range(len(n_coloring)):
    if not monochromatic_triangle_check(n_coloring[i]):
        print(n_coloring[i])
        print(i)
        break

In [ ]:
all_colorings=generate_all_colorings(5)

red = list(all_colorings[219][0])
blue = list(all_colorings[219][1])

r_coloring_K_n(5,edge_coloring=[red,blue], node_size=2000, labels={0:"Hobbes", 1:"Calvin", 2:"Scooby",3:"Jerry",4:"Linus"}, palette=["red","blue"])

In [ ]:
#Now we check K_6, which if you read the material you know this will not return edge partition which will meet our criteria
n_coloring = generate_all_colorings(6)
for i in range(len(n_coloring)):
    if not monochromatic_triangle_check(n_coloring[i]):
        print(n_coloring[i])
        print(i)
        break

In [ ]:
#For our more elegant algorithm, we need a danger score as well as our solving function, see site material for breakdown and commentary
def danger_score(u, v, color, coloring, n_clique):
    if n_clique <= 2:
        return 0
    colored_with_u = {
        w for (a, b), c in coloring.items()
        if c == color and (a == u or b == u)
        for w in ([b] if a == u else [a])
    }
    colored_with_v = {
        w for (a, b), c in coloring.items()
        if c == color and (a == v or b == v)
        for w in ([b] if a == v else [a])
    }
    W = frozenset(colored_with_u & colored_with_v)
    return count_cliques(W, coloring, color, n_clique - 2)



def solve(N,edges, idx, coloring, r, n_clique, stats, save_dir=None):

    stats['nodes'] += 1
    partitions = {c: [] for c in range(1, r + 1)}
    partitions[0] = [] 
    for e in edges[:idx]:
        u2, v2 = e
        partitions[coloring[_edge(u2, v2)]].append(e)
    for e in edges[idx:]:
        partitions[0].append(e)
    print(f"\n--- Node {stats['nodes']} | Edge idx {idx}/{len(edges)} ---")
    for c in range(1, r + 1):
        print(f"  {COLOR_NAMES[c-1]:8}: {partitions[c]}")
    print(f"  {'Uncolored':8}: {partitions[0]}")
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        N = max(v for e in edges for v in e) + 1
        edge_coloring = [partitions[c] for c in range(1, r + 1)]
        edge_coloring.append(partitions[0])         
        palette = r_colors[:r] + ["lightgray"]        
        node = stats['nodes']
        all_labs = {0:"Hobbes",1:"Calvin",2:"Scooby",3:"Jerry",4:"Linus",5:"Verdi"}
        labels = {key: all_labs[key] for key in range(N) if key in all_labs}
        title = (f"Step {node} | {idx}/{len(edges)} edges colored"
                 f"{'  ✓ complete' if idx == len(edges) else ''}")
        save_path = os.path.join(save_dir, f"node_{node:04d}.jpg")
        r_coloring_K_n(
            n=N,
            edge_coloring=edge_coloring,
            palette=palette,
            labels=labels,
            title=title,
            save_path=save_path,
            node_size=2000
        )
    if idx == len(edges):
        return coloring
    u, v = edges[idx]
    edge = _edge(u, v)
    color_order = sorted(
        range(1, r + 1),
        key=lambda c: danger_score(u, v, c, coloring, n_clique)
    )
    for color in color_order:
        if not is_forbidden(u, v, color, coloring, n_clique):
            coloring[edge] = color                          
            result = solve(N,edges, idx + 1, coloring, r, n_clique, stats, save_dir)
            if result is not None:
                return result                                
            del coloring[edge]                             
            stats['backtracks'] += 1
    return None  

In [ ]:
#This produces EACH STEP in the solver, so for K_3 it's pretty short, K_4 still manageable, K_5 not too bad, but K_6 gives a ton of steps and images... so beware!
solve(5,make_edges_by_vertex(5), 0, {}, 2, 3, {'nodes': 0, 'backtracks': 0})

In [ ]:
#This produces EACH STEP in the solver, so for K_3 it's pretty short, K_4 still manageable, K_5 not too bad, but K_6 gives a ton of steps and images... so beware!
solve(6,make_edges_by_vertex(6), 0, {}, 2, 3, {'nodes': 0, 'backtracks': 0})

In [ ]:
#Here are some pretty graphs with random colorings. Find one without a monochromatic triangle and you'll win a big prize!
edges = list(combinations(range(43),2))


red = {e for e in edges if random.randint(0,1)==1}
blue = set(edges)-red

r_coloring_K_n(43,title=r"$K_{43}$",edge_coloring=[red,blue],with_labels=False, node_size=50, palette=["red","blue"],width=0.2)

edges = list(combinations(range(46),2))


red = {e for e in edges if random.randint(0,1)==1}
blue = set(edges)-red

r_coloring_K_n(46,title=r"$K_{46}$",edge_coloring=[red,blue],with_labels=False, node_size=50, palette=["red","blue"],width=0.2)